In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Simplified PPS Audio Generator

This script reads participant-specific design CSV files from the ExperimentLog
directory and generates corresponding looming and tactile audio files.

The workflow:
1. Read the design CSV files for each participant
2. Load the looming noise and tactile stimulus files
3. Create two output files per participant:
   - Looming audio: Box breathing with looming stimuli at specified timestamps
   - Tactile audio: Empty audio with tactile stimuli at specified timestamps
4. Save the output files in the ExperimentAudio directory

Author: AI Assistant
"""

import os
import numpy as np
import pandas as pd
import soundfile as sf
import re
import logging
import glob
import datetime
import traceback

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# ----------------------------------------------------------------------
# CONFIGURATION PARAMETERS
# ----------------------------------------------------------------------
CONFIG = {
    'paths': {
        # Input directories
        'design_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog",
        'stimulus_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles",
        # Output directory
        'output_dir': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio",
        # Base audio file (for duration and timing reference)
        'base_audio': r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\BoxBreathing_complete.wav",
    },
    'stimulus': {
        # Looming stimulus files - should match the names in design CSVs
        'looming': {
            'blue': 'front_az0_FABIAN_HRIR_blue.wav',
            'brown': 'front_az0_FABIAN_HRIR_brown.wav',
            'pink': 'front_az0_FABIAN_HRIR_pink.wav',
            'white': 'front_az0_FABIAN_HRIR_white.wav',
        },
        # Tactile stimulus file
        'tactile': 'tactile_stimulus.wav',
    },
    'debug_mode': True,  # Enable detailed logging
    'sample_rate': 48000, # Fallback sample rate if not specified in files
}

# ----------------------------------------------------------------------
# Helper Function
# ----------------------------------------------------------------------
def parse_timestamp(timestamp):
    """
    Parse timestamp in format MM:SS.S to milliseconds.
    
    Args:
        timestamp: String in format "MM:SS.S"
        
    Returns:
        Float: milliseconds or None if timestamp is invalid
    """
    if pd.isna(timestamp):
        return None
        
    match = re.match(r'(\d+):(\d+\.\d+)', timestamp)
    if match:
        minutes, seconds = match.groups()
        return (int(minutes) * 60 + float(seconds)) * 1000
    return None

# ----------------------------------------------------------------------
# Main Audio Generator Class
# ----------------------------------------------------------------------
class PPSAudioGenerator:
    """Generates participant-specific audio files for PPS experiments."""
    
    def __init__(self, config=None):
        self.config = config or CONFIG
        
        # Setup directories
        self.design_dir = self.config['paths']['design_dir']
        self.stimulus_dir = self.config['paths']['stimulus_dir']
        self.output_dir = self.config['paths']['output_dir']
        
        # Create output directory if it doesn't exist
        os.makedirs(self.output_dir, exist_ok=True)
        
        # Find the parent audio file - looking for any file matching BoxBreathing_complete*.wav
        self.base_audio_path = self._find_parent_audio()
        
        if self.base_audio_path and os.path.exists(self.base_audio_path):
            self.base_audio, self.sample_rate = sf.read(self.base_audio_path)
            print(f"Loaded parent audio: {self.base_audio_path}")
            print(f"  Duration: {len(self.base_audio) / self.sample_rate:.2f} seconds")
            print(f"  Shape: {self.base_audio.shape}")
        else:
            # This is a critical error - we need the parent audio
            self.base_audio = None
            self.sample_rate = self.config.get('sample_rate', 48000)
            print(f"ERROR: Parent audio file not found in {self.output_dir}")
            print(f"The looming stimuli MUST be overlaid onto the parent audio file!")
            print(f"Please run the design generator first to create BoxBreathing_complete*.wav.")
        
        # Load stimulus files
        self.looming_stimuli = self._load_looming_stimuli()
        self.tactile_stimulus = self._load_tactile_stimulus()
    
    def _find_parent_audio(self):
        """Find the parent audio file by looking for BoxBreathing_complete*.wav"""
        pattern = os.path.join(self.output_dir, "BoxBreathing_complete*.wav")
        matches = glob.glob(pattern)
        
        if matches:
            # Use the first match (there should only be one)
            return matches[0]
        
        # Try the explicit path from config as a backup
        if os.path.exists(self.config['paths']['base_audio']):
            return self.config['paths']['base_audio']
            
        return None
    
    def _load_looming_stimuli(self):
        """Load all looming stimulus files."""
        looming_stimuli = {}
        
        stim_config = self.config['stimulus']['looming']
        for stim_type, filename in stim_config.items():
            file_path = os.path.join(self.stimulus_dir, filename)
            if os.path.exists(file_path):
                try:
                    data, sr = sf.read(file_path)
                    if sr != self.sample_rate:
                        print(f"Warning: Sample rate mismatch: {file_path} ({sr} Hz vs {self.sample_rate} Hz)")
                    
                    looming_stimuli[stim_type] = {
                        'data': data, 
                        'sample_rate': sr,
                        'duration': len(data) / sr,
                        'path': file_path
                    }
                    print(f"Loaded looming stimulus: {stim_type} ({len(data) / sr:.2f} seconds)")
                except Exception as e:
                    print(f"Error loading looming stimulus {stim_type} ({file_path}): {e}")
            else:
                print(f"Looming stimulus file not found: {file_path}")
        
        if not looming_stimuli:
            print("No looming stimuli loaded!")
        
        return looming_stimuli
    
    def _load_tactile_stimulus(self):
        """Load tactile stimulus file."""
        tactile_path = os.path.join(self.stimulus_dir, self.config['stimulus']['tactile'])
        
        if os.path.exists(tactile_path):
            try:
                data, sr = sf.read(tactile_path)
                if sr != self.sample_rate:
                    print(f"Warning: Sample rate mismatch: {tactile_path} ({sr} Hz vs {self.sample_rate} Hz)")
                
                tactile_info = {
                    'data': data,
                    'sample_rate': sr,
                    'duration': len(data) / sr,
                    'path': tactile_path
                }
                print(f"Loaded tactile stimulus: {tactile_path} ({len(data) / sr:.2f} seconds)")
                return tactile_info
            except Exception as e:
                print(f"Error loading tactile stimulus: {e}")
        else:
            print(f"Tactile stimulus file not found: {tactile_path}")
        
        return None
    
    def find_participant_design_files(self):
        """Find all participant design CSV files in the design directory."""
        design_files = []
        
        # Look for files matching the pattern: participant_X_design.csv
        pattern = os.path.join(self.design_dir, "participant_*_design.csv")
        
        for file_path in glob.glob(pattern):
            # Extract participant ID from filename
            match = re.search(r'participant_(\d+)_design\.csv', os.path.basename(file_path))
            if match:
                participant_id = int(match.group(1))
                design_files.append((participant_id, file_path))
        
        # Sort by participant ID
        design_files.sort(key=lambda x: x[0])
        
        if design_files:
            print(f"Found {len(design_files)} participant design files")
        else:
            print(f"No participant design files found in {self.design_dir}")
        
        return design_files
    
    def load_participant_design(self, participant_id, design_path):
        """
        Load and parse participant design CSV.
        
        Args:
            participant_id: Participant ID number
            design_path: Path to design CSV file
            
        Returns:
            Processed design data dictionary
        """
        try:
            # Load design CSV
            design_df = pd.read_csv(design_path)
            
            # Parse timestamps for looming and tactile stimuli
            looming_trials = []
            tactile_trials = []
            
            for _, row in design_df.iterrows():
                trial_num = row['trial_number']
                trial_type = row['trial_type']
                
                # Skip practice trials
                if trial_type == 'practice':
                    continue
                
                # Process looming stimulus for non-baseline trials
                if trial_type != 'baseline' and 'looming_stimulus_timestamp' in row:
                    looming_ts = row['looming_stimulus_timestamp']
                    looming_ms = parse_timestamp(looming_ts)
                    
                    if looming_ms is not None:
                        looming_trial = {
                            'trial_number': trial_num,
                            'trial_type': trial_type,
                            'timestamp_ms': looming_ms,
                            'stimulus_type': row.get('stimulus_type', 'front'),  # Default to front if not specified
                        }
                        looming_trials.append(looming_trial)
                
                # Process tactile stimulus for non-catch trials
                if trial_type != 'catch' and 'tactile_stimulus_timestamp' in row:
                    tactile_ts = row['tactile_stimulus_timestamp']
                    tactile_ms = parse_timestamp(tactile_ts)
                    
                    if tactile_ms is not None:
                        tactile_trial = {
                            'trial_number': trial_num,
                            'trial_type': trial_type,
                            'timestamp_ms': tactile_ms,
                        }
                        tactile_trials.append(tactile_trial)
            
            # Sort by timestamp
            looming_trials.sort(key=lambda x: x['timestamp_ms'])
            tactile_trials.sort(key=lambda x: x['timestamp_ms'])
            
            print(f"Loaded design for participant {participant_id}:")
            print(f"  - Looming trials: {len(looming_trials)}")
            print(f"  - Tactile trials: {len(tactile_trials)}")
            
            return {
                'participant_id': participant_id,
                'design_path': design_path,
                'looming_trials': looming_trials,
                'tactile_trials': tactile_trials,
            }
            
        except Exception as e:
            print(f"Error loading design for participant {participant_id}: {e}")
            traceback.print_exc()
            return None
    
    def create_participant_audio(self, design_data):
        """
        Create participant-specific audio files by placing stimuli at specified timestamps.
        
        Args:
            design_data: Dictionary with processed design data
            
        Returns:
            Tuple of (looming_path, tactile_path)
        """
        participant_id = design_data['participant_id']
        looming_trials = design_data['looming_trials']
        tactile_trials = design_data['tactile_trials']
        
        print(f"Creating audio for participant {participant_id}")
        
        # CRITICAL: For looming audio, use a copy of the base audio file (breathing guide)
        # This ensures the looming stimuli are overlaid onto the breathing audio
        if self.base_audio is not None:
            # Make a copy of the parent audio for looming
            looming_audio = np.copy(self.base_audio)
            duration_samples = len(looming_audio)
            audio_shape = looming_audio.shape
            print(f"  - Using parent audio as base for looming file")
        else:
            # Fallback if base audio not available (should not happen)
            print(f"  - ERROR: Parent audio file not found! Cannot create proper looming audio.")
            # Calculate based on last timestamp plus margin
            last_looming = max(t['timestamp_ms'] for t in looming_trials) if looming_trials else 0
            last_tactile = max(t['timestamp_ms'] for t in tactile_trials) if tactile_trials else 0
            last_timestamp = max(last_looming, last_tactile)
            
            # Add 30 seconds margin
            duration_sec = (last_timestamp / 1000) + 30
            duration_samples = int(duration_sec * self.sample_rate)
            
            # Determine audio shape (mono or stereo)
            if self.looming_stimuli and len(next(iter(self.looming_stimuli.values()))['data'].shape) > 1:
                # Stereo
                looming_audio = np.zeros((duration_samples, 2), dtype=np.float32)
                audio_shape = (duration_samples, 2)
            else:
                # Mono
                looming_audio = np.zeros(duration_samples, dtype=np.float32)
                audio_shape = (duration_samples,)
        
        # Create empty array for tactile audio (always same shape as looming)
        if len(audio_shape) > 1:
            # Stereo
            tactile_audio = np.zeros((duration_samples, audio_shape[1]), dtype=np.float32)
        else:
            # Mono
            tactile_audio = np.zeros(duration_samples, dtype=np.float32)
        
        print(f"  - Audio duration: {duration_samples / self.sample_rate:.2f} seconds")
        
        # Insert looming stimuli
        looming_inserted = 0
        for trial in looming_trials:
            trial_num = trial['trial_number']
            timestamp_ms = trial['timestamp_ms']
            stim_type = trial['stimulus_type']
            
            # Convert timestamp to sample index
            start_sample = int(timestamp_ms * self.sample_rate / 1000)
            
            # Get stimulus data
            if stim_type in self.looming_stimuli:
                stim_data = self.looming_stimuli[stim_type]['data']
                end_sample = start_sample + len(stim_data)
                
                # Check if it fits within array bounds
                if end_sample <= len(looming_audio):
                    # Insert stimulus (handle mono/stereo differences)
                    if len(looming_audio.shape) > 1 and len(stim_data.shape) > 1:
                        # Both stereo
                        looming_audio[start_sample:end_sample] += stim_data
                    elif len(looming_audio.shape) > 1 and len(stim_data.shape) == 1:
                        # Output stereo, stimulus mono
                        for ch in range(looming_audio.shape[1]):
                            looming_audio[start_sample:end_sample, ch] += stim_data
                    elif len(looming_audio.shape) == 1 and len(stim_data.shape) > 1:
                        # Output mono, stimulus stereo
                        looming_audio[start_sample:end_sample] += stim_data.mean(axis=1)
                    else:
                        # Both mono
                        looming_audio[start_sample:end_sample] += stim_data
                    
                    looming_inserted += 1
                else:
                    print(f"  - Warning: Trial {trial_num}: Looming stimulus exceeds audio length")
            else:
                print(f"  - Warning: Trial {trial_num}: Looming stimulus type '{stim_type}' not found")
        
        # Insert tactile stimuli
        tactile_inserted = 0
        for trial in tactile_trials:
            trial_num = trial['trial_number']
            timestamp_ms = trial['timestamp_ms']
            
            # Convert timestamp to sample index
            start_sample = int(timestamp_ms * self.sample_rate / 1000)
            
            # Get stimulus data
            if self.tactile_stimulus:
                stim_data = self.tactile_stimulus['data']
                end_sample = start_sample + len(stim_data)
                
                # Check if it fits within array bounds
                if end_sample <= len(tactile_audio):
                    # Insert stimulus (handle mono/stereo differences)
                    if len(tactile_audio.shape) > 1 and len(stim_data.shape) > 1:
                        # Both stereo
                        tactile_audio[start_sample:end_sample] += stim_data
                    elif len(tactile_audio.shape) > 1 and len(stim_data.shape) == 1:
                        # Output stereo, stimulus mono
                        for ch in range(tactile_audio.shape[1]):
                            tactile_audio[start_sample:end_sample, ch] += stim_data
                    elif len(tactile_audio.shape) == 1 and len(stim_data.shape) > 1:
                        # Output mono, stimulus stereo
                        tactile_audio[start_sample:end_sample] += stim_data.mean(axis=1)
                    else:
                        # Both mono
                        tactile_audio[start_sample:end_sample] += stim_data
                    
                    tactile_inserted += 1
                else:
                    print(f"  - Warning: Trial {trial_num}: Tactile stimulus exceeds audio length")
            else:
                print(f"  - Warning: Trial {trial_num}: Tactile stimulus not available")
        
        # Normalize audio if needed (prevent clipping)
        max_looming = np.max(np.abs(looming_audio))
        if max_looming > 0.95:
            scale_factor = 0.95 / max_looming
            looming_audio *= scale_factor
            print(f"  - Scaled looming audio by {scale_factor:.4f} to prevent clipping")
        
        max_tactile = np.max(np.abs(tactile_audio))
        if max_tactile > 0.95:
            scale_factor = 0.95 / max_tactile
            tactile_audio *= scale_factor
            print(f"  - Scaled tactile audio by {scale_factor:.4f} to prevent clipping")
        
        # Save audio files
        looming_filename = f"participant_{participant_id}_design_looming.wav"
        tactile_filename = f"participant_{participant_id}_design_tactile.wav"
        
        looming_path = os.path.join(self.output_dir, looming_filename)
        tactile_path = os.path.join(self.output_dir, tactile_filename)
        
        try:
            sf.write(looming_path, looming_audio, self.sample_rate)
            sf.write(tactile_path, tactile_audio, self.sample_rate)
            
            print(f"  - Successfully saved audio files:")
            print(f"    * Looming ({looming_inserted} stimuli): {looming_path}")
            print(f"    * Tactile ({tactile_inserted} stimuli): {tactile_path}")
            
            return looming_path, tactile_path
            
        except Exception as e:
            print(f"  - Error saving audio files: {e}")
            traceback.print_exc()
            return None, None
    
    def process_participant(self, participant_id, design_path):
        """
        Process a single participant: load design and create audio files.
        
        Args:
            participant_id: Participant ID number
            design_path: Path to design CSV file
            
        Returns:
            Dictionary with processing results
        """
        try:
            print(f"\n=== Processing participant {participant_id} ===")
            
            # Load design
            design_data = self.load_participant_design(participant_id, design_path)
            if not design_data:
                return {
                    'participant_id': participant_id,
                    'success': False,
                    'error': 'Failed to load design data'
                }
            
            # Create audio files
            looming_path, tactile_path = self.create_participant_audio(design_data)
            if not looming_path or not tactile_path:
                return {
                    'participant_id': participant_id,
                    'success': False,
                    'error': 'Failed to create audio files'
                }
            
            return {
                'participant_id': participant_id,
                'success': True,
                'looming_path': looming_path,
                'tactile_path': tactile_path
            }
            
        except Exception as e:
            print(f"Error processing participant {participant_id}: {e}")
            traceback.print_exc()
            return {
                'participant_id': participant_id,
                'success': False,
                'error': str(e)
            }
    
    def run(self):
        """
        Process all participant design files and generate audio.
        
        Returns:
            Dictionary with processing results
        """
        # Find participant design files
        design_files = self.find_participant_design_files()
        
        if not design_files:
            print("No participant design files found. Exiting.")
            return {'success': False, 'error': 'No design files found'}
        
        results = {}
        start_time = datetime.datetime.now()
        
        for i, (participant_id, design_path) in enumerate(design_files):
            print(f"Processing participant {participant_id} ({i+1}/{len(design_files)})")
            result = self.process_participant(participant_id, design_path)
            results[participant_id] = result
        
        # Calculate processing time
        elapsed_time = (datetime.datetime.now() - start_time).total_seconds()
        
        # Count successes and failures
        successes = sum(1 for r in results.values() if r.get('success', False))
        failures = len(results) - successes
        
        print(f"\n=== Processing completed in {elapsed_time:.2f} seconds ===")
        print(f"Processed {len(results)} participants:")
        print(f"  - Successful: {successes}")
        print(f"  - Failed: {failures}")
        
        return {
            'success': True,
            'participant_results': results,
            'successful_count': successes,
            'failed_count': failures,
            'processing_time': elapsed_time
        }

# ----------------------------------------------------------------------
# MAIN FUNCTION
# ----------------------------------------------------------------------
def main():
    """Main function to run the audio generator."""
    print("\n===== PPS EXPERIMENT AUDIO GENERATOR =====")
    print(f"Design directory: {CONFIG['paths']['design_dir']}")
    print(f"Stimulus directory: {CONFIG['paths']['stimulus_dir']}")
    print(f"Output directory: {CONFIG['paths']['output_dir']}")
    
    # Create the generator (will automatically find parent audio)
    generator = PPSAudioGenerator(CONFIG)
    
    # Check if parent audio was found
    if generator.base_audio is None:
        print("\nERROR: Parent audio file not found!")
        print("Looking for a file named BoxBreathing_complete*.wav")
        print("in the ExperimentAudio directory. This file should have been")
        print("created by the PPS Design Generator.")
        print("\nPlease run the Design Generator first, then try again.")
        return
    
    # Run the generator
    results = generator.run()
    
    # Print summary
    print("\n===== GENERATION SUMMARY =====")
    if results.get('success', False):
        print(f"Successfully processed {results['successful_count']}/{results['successful_count'] + results['failed_count']} participants.")
        print(f"Processing time: {results['processing_time']:.2f} seconds")
    else:
        print(f"Generation failed: {results.get('error', 'Unknown error')}")

if __name__ == "__main__":
    main()


===== PPS EXPERIMENT AUDIO GENERATOR =====
Design directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentLog
Stimulus directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles
Output directory: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio
Loaded parent audio: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio\BoxBreathing_complete_x5.wav
  Duration: 1675.26 seconds
  Shape: (80412672, 2)
Loaded looming stimulus: blue (3.00 seconds)
Loaded looming stimulus: brown (3.00 seconds)
Loaded looming stimulus: pink (3.00 seconds)
Loaded looming stimulus: white (3.00 seconds)
Loaded tactile stimulus: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\tactile_stimulus.wav (1.00 seconds)
Found 100 participant design files
Processing participant 0 (1/100)

=== Processing participant 0 ===
Loaded design for participa